## Precision and Recall

**Precision:** of everything the model flagged, how many were actually correct?  
**Recall:** of everything that was actually correct, how many did the model catch?

## How it actually works

Imagine 120 fraudulent invoices exist in your system.  
Your model flags 100 invoices as fraudulent.  
Out of those 100 flagged — only 70 are actually fraud. 30 are legitimate invoices wrongly flagged.

**Precision = 70 out of 100 = 70%**  
Of everything the model said was fraud, 70% actually was.  
The other 30% were false alarms.

**Recall = 70 out of 120 = 58%**  
Of all the real fraud that existed, the model only caught 58%.  
42% of actual fraud slipped through undetected.

A model can be high precision but low recall — very accurate when it flags something,  
but misses a lot of real cases.  
Or high recall but low precision — catches almost everything real,  
but drowns you in false alarms.  
You rarely get both high at the same time. That tension is the core challenge.

## Where it breaks or gets dangerous

**The precision-recall tradeoff:**  
Make the model more aggressive (lower threshold to flag) →  
recall goes up, precision drops. More real fraud caught, more false alarms generated.  
Make the model more conservative (higher threshold) →  
precision goes up, recall drops. Cleaner alerts, but real fraud slips through.

**The dangerous mistake:**  
Optimising only for accuracy and ignoring precision and recall.  
A model that labels everything as "not fraud" on a dataset with 1% fraud rate  
will have 99% accuracy — and catch zero fraud.  
Accuracy alone is meaningless on imbalanced datasets.

**Who decides the tradeoff?**  
Not the data scientist. The business.  
Which is more costly — a false alarm that delays a legitimate payment,  
or a missed fraud case that costs the company money?  
That is a CFO decision, not a technical one.

In [ ]:
# Precision and Recall — illustration with invoice fraud scenario

def calculate_precision_recall(true_positives, false_positives, false_negatives):
    precision = true_positives / (true_positives + false_positives)
    recall = true_positives / (true_positives + false_negatives)
    return precision, recall

# Scenario from the explanation
total_actual_fraud = 120
model_flagged = 100
correctly_flagged = 70
false_alarms = model_flagged - correctly_flagged        # 30
missed_fraud = total_actual_fraud - correctly_flagged   # 50

precision, recall = calculate_precision_recall(
    true_positives=correctly_flagged,
    false_positives=false_alarms,
    false_negatives=missed_fraud
)

print(f"Total actual fraud cases:     {total_actual_fraud}")
print(f"Model flagged as fraud:        {model_flagged}")
print(f"Correctly flagged:             {correctly_flagged}")
print(f"False alarms:                  {false_alarms}")
print(f"Missed fraud cases:            {missed_fraud}")
print()
print(f"Precision: {precision:.0%}  — of what model flagged, this % was actually fraud")
print(f"Recall:    {recall:.0%}  — of all real fraud, model caught this %")
print()

# What happens when we make the model more aggressive
print("--- If model flags 200 invoices, catches 100 real fraud ---")
p2, r2 = calculate_precision_recall(100, 100, 20)
print(f"Precision: {p2:.0%}  Recall: {r2:.0%}  → more fraud caught, more false alarms")

# What happens when we make the model more conservative  
print("--- If model flags 50 invoices, catches 45 real fraud ---")
p3, r3 = calculate_precision_recall(45, 5, 75)
print(f"Precision: {p3:.0%}  Recall: {r3:.0%}  → cleaner alerts, more fraud missed")

## My D365 analogy

In D365 three-way matching — PO, receipt, invoice —  
the system flags mismatches for review.

**Precision problem:**  
Matching tolerance set too tight.  
System flags every minor rounding difference as a mismatch.  
AP team drowns in false alerts. Legitimate invoices get delayed.  
High recall, low precision.

**Recall problem:**  
Matching tolerance set too loose.  
System approves invoices that should have been flagged.  
Fraudulent or incorrect invoices get posted without review.  
High precision on what it does flag, but missing real problems.

Every D365 consultant has tuned matching tolerances.  
You were solving a precision-recall tradeoff without knowing the name for it.

## LinkedIn post idea

Every D365 consultant who has configured three-way matching  
has already solved a precision-recall problem — they just did not know it had a name.

Set the tolerance too tight: your AP team drowns in false alerts. High recall, low precision.  
Set it too loose: real mismatches slip through undetected. High precision, low recall.  
You find the balance that fits the business risk appetite.

That is exactly what data scientists do when tuning a machine learning model.  
The math is different. The business question is identical:  
which failure mode costs more — the false alarm or the missed case?

14 years of ERP configuration taught me this before I ever opened a ML textbook.  
Domain expertise is not separate from AI engineering. It is the foundation of it.

## Model Evaluation

How do you know if your model is actually good?  
You measure it. That process is called model evaluation.

Three concepts you must know:

**Train / Validation / Test split:**  
- Training set — model learns from this (typically 70-80% of data)
- Validation set — used during training to tune hyperparameters (10-15%)
- Test set — never touched until final evaluation (10-15%)  
The test set is sacred. If you evaluate on it during development, your results are lies.

**Accuracy:**  
Of all predictions, what percentage were correct?  
Simple but dangerous on imbalanced datasets.  
1% fraud rate → predicting "not fraud" always = 99% accuracy, zero value.

**F1 Score:**  
A single number that balances precision and recall.  
Useful when you cannot decide which matters more.  
F1 = 2 × (Precision × Recall) / (Precision + Recall)  
High F1 = model is good at both catching real cases and avoiding false alarms.

## Evaluation in D365 context

Before go-live, you run UAT — User Acceptance Testing.  
You test against scenarios the system has never been configured for.  
If it passes, you go live. If not, you fix and retest.

Model evaluation is UAT for machine learning.  
The test set is your UAT dataset.  
Accuracy, precision, recall, and F1 are your UAT pass/fail criteria.

The mistake most teams make:  
They evaluate the model on the same data it trained on.  
That is like running UAT using only the data you used to build the configuration.  
Of course it passes. It means nothing.

## LinkedIn post idea

In D365 implementations we never skip UAT.  
Everyone knows why — you cannot test a system using the same data you built it with.  
Real validation means scenarios the system has never seen.

Machine learning has the same rule.  
It is called train/test split.  
You hold back data the model never sees during training,  
then test on that data only at the very end.

Skip this step and your model looks perfect in development and fails in production.  
Same mistake. Different technology. Same consequences.

The principles of good enterprise implementation  
and the principles of good AI engineering  
are the same principles.